# Olentangy Data Science Day 2026

*A hands-on introduction to simulation, modeling and problem-solving.*

<br/>

<img src="images/bread_financial_logo.svg" width="200" />

## About Bread Financial

<img src="images/Screenshot Bread Financial General 01.png" width="100%" />

<img src="images/Screenshot Bread Financial General 02.png" width="100%" />

## About Me

Principal data scientist at **Bread Financial**.

Today I'll walk you through some fun experiments that show how data science works in practice.

## Today's Demo

- Explore classic probability puzzles through **simulation and visualization**
- Understand *tokens* and *embeddings* used in *LLMs*
- Design a marketing campaign for a local coffee shop

<br/>

> Data scientists use similar approaches every day to test ideas, validate assumptions, and discover insights.

<br/>

All code and notebooks: **[github.com/wl37/datascienceday2026](https://github.com/wl37/datascienceday2026)**

## Part 1: Simulation (Probability and Statistics)

### The Coin Flip — How random is random?

Let's simulate thousands of coin flips and see what the data tells us.

In [ ]:
import random
import matplotlib.pyplot as plt

def simulate_coin_flips(n: int) -> tuple[float, float]:
    flips = [random.choice(['H', 'T']) for _ in range(n)]
    heads = flips.count('H') / n * 100
    tails = flips.count('T') / n * 100
    return heads, tails

for n in [10, 100, 1000, 10_000]:
    h, t = simulate_coin_flips(n)
    print(f"{n:>6} flips → Heads: {h:.1f}%  Tails: {t:.1f}%")

### 🎂 Birthday Match

<img src="images/birthday_match.png" width="400" />

Raise your hand when your birthday falls in the group — and let's see if we can find a match!


| Round | Months |
|:-----:|:-------|
| 1 | **January – March** |
| 2 | **April – June** |
| 3 | **July – September** |
| 4 | **October – December** |


#### Re-frame the Same Question

Do you think a shared birthday among a room of people like this is,  

1.rare; 2. possible, or 3. actually likely?


OR, 
For a group of 23 people, would you guess the chance of a match is:

- below 25%?
- around 50%?
- above 75%?

#### Let's simulate!

We'll get back to the math — but there's a way to **touch & feel** it yourself. With AI and coding assistant, everyone can design and run simulations. How would you design a simulation for this problem?


What if we just **built the room** inside the computer? Give 23 people each a random birthday, then check whether any two match. Do that thousands of times and count how often a match happens.

In [ ]:
import random

def has_birthday_match(group_size: int) -> bool:
    """Simulate a room of people with random birthdays; return True if any two share one."""
    birthdays = [random.randint(1, 365) for _ in range(group_size)]
    return len(birthdays) != len(set(birthdays))


def estimate_match_probability(group_size: int, trials: int = 10_000) -> float:
    """Run the experiment many times and return the fraction that had a match."""
    matches = sum(has_birthday_match(group_size) for _ in range(trials))
    return matches / trials


for n in [10, 15, 20, 23, 30, 40, 50]:
    prob = estimate_match_probability(n)
    print(f"{n:>2} people  →  {prob * 100:5.1f}% chance of a shared birthday")

#### Why Is It So High?

The simulation gave us the answer — but *why* does a group of just 23 cross 50%? Let's look at the math behind what the computer just did.

**What is the probability that nobody matches?**

- The first person can have any birthday: $\frac{365}{365}$
- The second person must avoid that one day: $\frac{364}{365}$
- The third person must avoid two taken days: $\frac{363}{365}$
- And the pattern continues

So for a group of size $n$:

$$P(\text{no match}) = \frac{365}{365} \times \frac{364}{365} \times \frac{363}{365} \times \cdots \times \frac{365-n+1}{365}$$

But the original question asks for **at least one match**, so we flip the answer:

$$P(\text{at least one match}) = 1 - P(\text{no match})$$

For $n = 23$:

$$P(\text{at least one match}) \approx 0.507$$

That is just over **50%**.

#### Reflect

Why does it rise so quickly? Because the number of possible pairs grows fast:

$$\binom{n}{2} = \frac{n(n-1)}{2}$$

For 23 people, that is **253 pairs**. The room does not feel very large, but the number of chances for a match is already much bigger than most people expect.

### 🚪 The Monty Hall Problem

It's the 1970s. You're a contestant on *Let's Make a Deal*. The host, Monty Hall, stands in front of three closed doors.

Behind one door is a brand-new car. Behind the other two — goats.

You pick a door. Say, Door #1. But before you open it, Monty — who knows *exactly* what's behind every door — reveal another door that has a goat. 

Now only two doors remain: a car and a goat. Monty would give you one more chance:

> ***"Would you like to switch?"***

#### What Would You Do?

Two doors left. One car. Before we go any further, commit to an answer:

- **Stay** with your original door?
- **Switch** to the other one?
- **Doesn't matter** — it's the same either way?

Lock in your choice. Don't change it once you see what comes next.

#### Let's Play a Thousand Times - Simulation

Arguments won't settle this — but data will. How would you design a simulation for this game?


Think about what one round looks like:
1. Randomly place the car behind one of three doors.
2. The player picks a door.
3. Monty opens a door that is **not** the player's pick and **not** the car.
4. The player either stays or switches.
5. Record whether they won.

Now repeat that thousands of times — once for a player who always stays, and once for a player who always switches. Then compare.

In [11]:
import random

def run_monty_hall(num_trials: int, switch_door: bool) -> int:
    """Play the Monty Hall game num_trials times and return total wins."""
    wins = 0
    doors = [0, 1, 2]

    for _ in range(num_trials):
        prize_door = random.choice(doors)
        player_pick = random.choice(doors)

        # Monty opens a door that is neither the prize nor the player's pick
        monty_opens = random.choice(
            [d for d in doors if d != prize_door and d != player_pick]
        )

        if switch_door:
            final = [d for d in doors if d != player_pick and d != monty_opens][0]
        else:
            final = player_pick

        if final == prize_door:
            wins += 1

    return wins


In [12]:
for n in [100, 1_000, 10_000]:
    stay_wins   = run_monty_hall(n, switch_door=False)
    switch_wins = run_monty_hall(n, switch_door=True)
    print(f"{n:>6} games  →  Stay: {stay_wins/n*100:5.1f}%   Switch: {switch_wins/n*100:5.1f}%")

   100 games  →  Stay:  34.0%   Switch:  61.0%
  1000 games  →  Stay:  33.8%   Switch:  67.4%
 10000 games  →  Stay:  33.2%   Switch:  67.6%


#### So Why Does Switching Work?

When you first picked a door, you had a **1-in-3** chance of being right. That means there was a **2-in-3** chance the car was behind one of the *other* doors.

Now here is the key: Monty is not choosing randomly. He *knows* where the car is and he *always* opens a goat door. He is giving you information. That entire $\frac{2}{3}$ probability collapses onto the single remaining door.

- **Stay** — you win only if your first guess was right — $\frac{1}{3}$
- **Switch** — you win whenever your first guess was *wrong* — $\frac{2}{3}$

Switching doesn't just help a little. It **doubles** your chances.

#### Reflect

This puzzle became famous in 1990 when Marilyn vos Savant published the answer in a magazine. Thousands of PhD holders wrote in to say she was wrong. She wasn't.

The lesson is the same one the birthday problem taught us: when your gut says one thing and the data says another, **the data wins**. And sometimes the best way to believe the math is to run the experiment yourself.